In [6]:
zipfile <- "HH_flux.zip"

In [7]:

setwd("/mnt/gsdata/projects/panops/panops-data-registry/data/flux")

In [ ]:
### load the libraries
library(data.table)
library(future)
library(stringr)
library(future.apply)

zipfile <- "HH_flux.zip"   # or "flux_HH.zip" if that’s the real name

wanted_cols <- c(
  "TIMESTAMP_START", "TIMESTAMP_END",
  "TA_F", "TA_F_QC", "VPD_F", "VPD_F_QC", "P_F", "P_F_QC",
  "TS_F_MDS_1", "TS_F_MDS_2", "TS_F_MDS_1_QC", "TS_F_MDS_2_QC",
  "WS_F", "WS_F_QC",
  "SWC_F_MDS_1", "SWC_F_MDS_2", "SWC_F_MDS_1_QC", "SWC_F_MDS_2_QC",
  "H_CORR", "H_RANDUNC", "H_RANDUNC_N", "NEE_VUT_REF_RANDUNC_N",
  "LE_F_MDS", "LE_F_MDS_QC", "LE_CORR", "LE_RANDUNC", "LE_RANDUNC_N",
  "H_F_MDS", "H_F_MDS_QC", "CO2_F_MDS", "CO2_F_MDS_QC",
  "NEE_VUT_REF", "NEE_VUT_REF_QC", "NEE_VUT_50", "NEE_VUT_50_QC",
  "NEE_VUT_REF_RANDUNC",
  "GPP_NT_VUT_REF", "GPP_DT_VUT_REF", "GPP_NT_VUT_50", "GPP_DT_VUT_50",
  "RECO_NT_VUT_REF", "RECO_DT_VUT_REF", "RECO_NT_VUT_50", "RECO_DT_VUT_50",
  "SW_IN_F", "SW_IN_F_QC", "PPFD_IN", "PPFD_DIF", "PPFD_OUT",
  "TA_F_MDS", "TA_F_MDS_QC", "VPD_F_MDS",
  "SW_IN_F_MDS", "SW_IN_F_MDS_QC", "SW_IN_POT",
  "LW_IN_F", "LW_OUT", "SW_OUT", "NETRAD",
  "USTAR", "WS", "G_F_MDS", "G_F_MDS_QC", "RH", "NIGHT", "PA"
)

outdir <- "merged_sites_subset"
dir.create(outdir, showWarnings = FALSE)

data.table::setDTthreads(2)

# list files in zip, no extraction
zip_files <- unzip(zipfile, list = TRUE)
csv_files <- zip_files$Name[
  grepl("\\.csv$", zip_files$Name) &
  !grepl("^__MACOSX/", zip_files$Name)
]

extract_site <- function(x) {
  str_extract(basename(x), "[A-Z]{2}-[A-Za-z0-9]+")
}

site_ids     <- extract_site(csv_files)
unique_sites <- unique(site_ids[!is.na(site_ids)])

length(unique_sites)
head(unique_sites)

# robust streaming reader
safe_fread_cmd <- function(cmd) {
  fread(cmd = cmd, showProgress = FALSE, fill = Inf)
}

read_file_from_zip <- function(fname_in_zip) {
  cmd <- sprintf("unzip -p %s %s | tr -d '\\000'",
                 shQuote(zipfile), shQuote(fname_in_zip))
  dt <- safe_fread_cmd(cmd)
  
  if (nrow(dt) > 0) {
    col1 <- dt[[1]]
    if (is.character(col1)) {
      bad  <- grepl("^Mac OS X", col1, useBytes = TRUE) |
              grepl("com\\.apple\\.quarantine", col1, useBytes = TRUE)
      if (any(bad)) dt <- dt[!bad]
    }
  }
  
  dt
}





[1] 521

[1] "US-Wi0" "ZA-Kru" "US-xWR" "AU-ASM" "CA-Gro" "JP-BBY"

Function to process a single site

This will:
	1.	Find all files for that site.
	2.	Read each file from the zip.
	3.	Keep only wanted_cols that actually exist in that file.
	4.	Row-bind all those pieces.
	5.	Remove duplicate rows for the same timestamp(s).
	6.	Write one <SITE>_merged.csv in merged_sites_subset.

In [10]:
process_one_site <- function(site) {
  tryCatch({
    message("=== Processing site: ", site, " ===")
    
    site_files <- csv_files[site_ids == site]
    if (length(site_files) == 0L) {
      message("   no files found in zip for site ", site)
      return(FALSE)
    }
    
    dt_list <- vector("list", length(site_files))
    
    for (i in seq_along(site_files)) {
      f <- site_files[i]
      message("   reading file: ", f)
      
      dt <- read_file_from_zip(f)
      if (nrow(dt) == 0L) next
      
      keep <- intersect(wanted_cols, names(dt))
      if (length(keep) == 0L) next
      
      dt_list[[i]] <- dt[, ..keep]
    }
    
    dt_list <- Filter(function(x) !is.null(x) && nrow(x) > 0L, dt_list)
    if (length(dt_list) == 0L) {
      message("   no usable data for site ", site)
      return(FALSE)
    }
    
    dt_all <- rbindlist(dt_list, use.names = TRUE, fill = TRUE)
    rm(dt_list); gc()
    
    key_cols <- intersect(c("TIMESTAMP_START", "TIMESTAMP_END"), names(dt_all))
    if (length(key_cols) > 0L) {
      setkeyv(dt_all, key_cols)
      dt_all <- unique(dt_all)
    } else {
      dt_all <- unique(dt_all)
    }
    
    out_path <- file.path(outdir, paste0(site, "_merged.csv"))
    if (file.exists(out_path)) file.remove(out_path)
    
    fwrite(dt_all, out_path)
    
    rm(dt_all); gc()
    TRUE
  }, error = function(e) {
    message("   ERROR for site ", site, ": ", conditionMessage(e))
    FALSE
  })
}

Parallel processing on HPC (without eating 128 cores)

In [ ]:
# polite parallelism
data.table::setDTthreads(2)
n_workers <- 16
plan(multisession, workers = n_workers)

# restartable: skip sites that already have a merged file
existing   <- list.files(outdir, pattern = "_merged\\.csv$", full.names = FALSE)
sites_done <- extract_site(existing)
sites_to_do <- setdiff(unique_sites, sites_done)

length(sites_to_do)

res <- future_lapply(sites_to_do, process_one_site)

table(unlist(res))

[1] 503

Quick check

In [6]:
table(site_ids)

site_ids
AR-Bal AR-CCa AR-CCg AR-SLu AR-TF1 AR-TF2 AR-Vir AT-Neu AU-Ade AU-Adr AU-ASM 
     2      2      2      2      2      2      2      2      2      2      2 
AU-Boy AU-Cpr AU-Cum AU-DaP AU-DaS AU-Dry AU-Emr AU-Fog AU-Gin AU-GWW AU-How 
     2      4      4      2      2      2      2      2      2      2      2 
AU-Lit AU-Lon AU-Lox AU-RDF AU-Rgf AU-Rig AU-Rob AU-Stp AU-TTE AU-Tum AU-Wac 
     2      2      2      2      2      2      2      4      2      2      2 
AU-War AU-Whr AU-Wom AU-Ync BE-Bra BE-Dor BE-Lcr BE-Lon BE-Maa BE-Vie BR-CST 
     2      2      2      2      4      4      2      4      4      4      2 
BR-Npw BR-Sa1 BR-Sa3 CA-ARB CA-ARF CA-BOU CA-Ca1 CA-Ca2 CA-Cbo CA-CF1 CA-DB2 
     2      2      2      2      2      2      2      2      2      2      2 
CA-DBB CA-DSM CA-EM1 CA-EM2 CA-ER1 CA-Gro CA-HPC CA-LP1 CA-MA1 CA-MA2 CA-MA3 
     2      2      2      2      2      2      2      2      2      2      2 
CA-Man CA-Mer CA-Mtk CA-NS1 CA-NS2 CA-NS3 CA-NS4 CA-NS5

In [7]:
View(unique_sites)
length(unique_sites)

[1] "US-Wi0" "ZA-Kru" "US-xWR" "AU-ASM" "CA-Gro" "JP-BBY" "US-Srr" "US-Fwf"
  [9] "CN-Cng" "US-Ho2" "US-RGo" "US-CS4" "US-ASM" "CR-Fsc" "GL-ZaF" "UK-AMo"
 [17] "AU-Rob" "FR-Lam" "AU-Wom" "US-SRM" "US-Myb" "NL-Loo" "AU-Rgf" "US-PFh"
 [25] "US-Me2" "US-Ro6" "US-PFb" "CA-NS7" "BR-Sa3" "US-Pnp" "US-Wi9" "US-Me5"
 [33] "US-xCP" "FR-Gri" "RU-Tks" "IT-Lav" "US-PFt" "US-PSL" "US-RGF" "DE-Kli"
 [41] "US-xBA" "US-xUN" "CN-Ha2" "SE-Deg" "SE-Svb" "AU-Lox" "US-PAS" "IT-Noe"
 [49] "ES-Pdu" "DK-RCW" "CA-Ca1" "CH-Fru" "FR-EM2" "US-UTV" "DE-RuS" "RU-Fy3"
 [57] "ES-LM2" "US-Ne1" "US-HB1" "US-DS3" "US-Vcm" "CA-TP4" "US-UMB" "CH-Aws"
 [65] "US-Tw5" "DE-Lkb" "US-HB2" "DE-Geb" "CZ-BK1" "DK-Sor" "CA-NS6" "DE-Hzd"
 [73] "IL-Yat" "GL-ZaH" "US-BRG" "US-IB2" "US-Wi6" "FR-Mej" "AR-TF1" "CA-MA2"
 [81] "US-xDJ" "US-Ne2" "RU-Cok" "US-Ro4" "US-xJR" "AU-Cpr" "PA-SPn" "CA-HPC"
 [89] "GH-Ank" "RU-Ha1" "US-MMS" "US-PFc" "US-xWD" "US-Ho1" "US-Me4" "AU-Dry"
 [97] "US-KFS" "US-RGW" "RU-SkP" "CA-NS1" "US-PFd" "US-Rwe" "US-NC4" "CA-DBB"
[105] "BE-Lon" "BE-Maa" "CA-EM1" "US-NC1" "DE-RuR" "US-Tw2" "DE-Zrk" "IT-BsB"
[113] "CA-SMC" "IT-MBo" "US-ICt" "CZ-KrP" "FR-Fon" "CL-SDP" "US-LA3" "US-Jo2"
[121] "DE-SfS" "AU-Emr" "US-CS6" "US-KS2" "DK-Skj" "US-Wi1" "US-CAK" "CA-MA3"
[129] "DE-Msr" "AU-Fog" "US-MOz" "US-A74" "US-CdM" "FI-Var" "US-MtB" "FR-Hes"
[137] "AU-Rig" "FI-Ken" "US-Bar" "CA-DB2" "CZ-Lnz" "US-DFC" "IE-CaN" "US-Me6"
[145] "FI-Let" "US-UMd" "CN-Din" "AR-Bal" "US-Hn3" "US-UTP" "US-DS1" "FI-Hyy"
[153] "CZ-wet" "US-MN2" "FR-FBn" "CA-Ca2" "US-KPL" "CD-Ygb" "US-DFK" "US-LS2"
[161] "CN-Qia" "FR-Aur" "CL-SDF" "US-xMB" "DE-Seh" "US-Fmf" "AR-SLu" "US-xST"
[169] "US-CF1" "US-xSE" "US-AR2" "AU-Lit" "US-CGG" "US-Mo1" "US-xSB" "ES-LJu"
[177] "IT-La2" "CA-TP1" "US-Twt" "FI-Sii" "IT-SRo" "CA-Mer" "IE-Cra" "IT-SR2"
[185] "CZ-RAJ" "DK-Fou" "US-BZo" "US-PFr" "AR-Vir" "AR-CCa" "DE-Akm" "US-xHE"
[193] "AU-GWW" "US-Esm" "US-CS5" "DE-Hai" "DE-Tha" "US-Wi4" "FI-Qvd" "AU-Wac"
[201] "IT-BCi" "US-PFn" "CA-DSM" "DE-Rns" "US-xJE" "US-Elm" "BE-Bra" "SE-Htm"
[209] "US-Tw3" "DE-Hdn" "CN-Dan" "US-BZF" "US-MN3" "US-Ton" "US-Fuf" "US-RRC"
[217] "US-UM3" "US-Seg" "FR-Bil" "US-Wi3" "AT-Neu" "US-NGC" "ES-LMa" "GF-Guy"
[225] "MY-PSO" "BE-Lcr" "US-UTJ" "CH-Oe1" "US-xDL" "US-xSR" "IE-Gwr" "US-Ro2"
[233] "US-xSC" "US-xRN" "US-GBT" "CA-Obs" "US-Wi5" "CA-SF2" "US-UC2" "FR-LGt"
[241] "US-Cop" "GL-Dsk" "DE-Gri" "IT-PT1" "CA-TPD" "US-Syv" "DE-Hte" "ES-LM1"
[249] "ES-Amo" "AU-Cum" "GL-NuF" "IT-Bsn" "CL-ACF" "CA-Man" "PA-SPs" "US-SRS"
[257] "US-Wjs" "US-RGB" "US-Kon" "US-WCr" "SJ-Blv" "CA-SF1" "US-Sag" "US-Ro3"
[265] "CA-SCC" "CA-ARB" "DE-RuC" "RU-Sam" "US-A39" "IT-Ro1" "US-xBL" "IT-CA3"
[273] "US-PFe" "US-xTR" "US-xCL" "US-Rwf" "CA-Qc2" "CH-Cha" "CG-Tch" "DE-Spw"
[281] "DE-RuW" "US-PFa" "US-Rpf" "US-Vcp" "IT-TrF" "US-SP1" "US-AR1" "US-UTB"
[289] "CZ-Stn" "IT-MtM" "US-OWC" "AU-TTE" "ES-Ln2" "IT-Lsn" "CA-TP2" "CH-Dav"
[297] "CA-TP3" "BE-Vie" "DK-Vng" "CH-Oe2" "US-MN1" "US-ARM" "CN-Du3" "US-BZB"
[305] "CN-Sw2" "US-SRG" "BE-Dor" "RU-Fyo" "IT-Niv" "US-Sta" "IE-CaC" "US-Slt"
[313] "IT-Ren" "US-Wi8" "CA-SF3" "CH-Lae" "CA-EM2" "US-xKA" "ES-LgS" "CA-SCB"
[321] "US-PFm" "US-Tw1" "AU-Adr" "US-PFg" "AU-Boy" "US-xAB" "US-xHA" "US-Me1"
[329] "US-Wi7" "IT-BFt" "US-xTA" "CN-Du2" "AU-RDF" "US-YK2" "CZ-BK2" "US-CRK"
[337] "US-Snf" "FR-CLt" "US-ICh" "ZM-Mon" "US-xBN" "US-Tw4" "CA-LP1" "US-xSL"
[345] "FI-Sod" "US-Mo3" "US-CS8" "AU-Stp" "ES-Agu" "FR-Lus" "IT-Isp" "US-UC1"
[353] "US-ORv" "JP-SMF" "EE-Pts" "US-CF4" "US-Hn2" "RU-Che" "DE-Har" "US-Rms"
[361] "IT-CA1" "US-xNG" "IT-MtP" "US-Ne3" "US-CF3" "US-DS2" "US-Cst" "US-xNQ"
[369] "AU-Ade" "US-Rls" "FR-Pue" "ES-Abr" "US-Var" "AU-DaS" "US-CS3" "CA-ARF"
[377] "US-Dmg" "US-StJ" "IT-Tor" "US-PFL" "FI-Lom" "US-EML" "CA-Qfo" "US-Rws"
[385] "US-CF2" "US-Blo" "US-Goo" "US-PFk" "SE-Nor" "IT-Col" "US-RGA" "IT-Ro2"
[393] "NL-Hor" "US-BZS" "US-Prr" "AR-CCg" "FR-Tou" "US-KS1" "DK-Gds" "US-KS3"
[401] "CA-ER1" "US-Bi1" "US-Ses" "AU-Tum" "CA-Cbo" "US-Wi2" "BR-Sa1" "US-A37"
[409] "US-Jo1" "IT-Cpz" 

[1] 521

Loop through each site and merge all datasets
(522 unique sites are exist)

In [37]:
outdir <- "merged_sites"
dir.create(outdir, showWarnings = FALSE)

data.table::setDTthreads(8)  # be nice to the server

safe_fread_file <- function(path) {
  cmd <- sprintf("tr -d '\\000' < %s", shQuote(path))
  fread(cmd = cmd, showProgress = FALSE, fill = TRUE)
}

for (site in unique_sites) {
  
  message("Processing: ", site)
  
  site_files  <- csv_files[site_ids == site]
  merged_path <- file.path(outdir, paste0(site, "_merged.csv"))
  first_write <- TRUE
  
  for (f in site_files) {
    
    # extract only this file
    unzip(zipfile, files = f, exdir = tempdir(), overwrite = TRUE)
    full_path <- file.path(tempdir(), f)
    
    # read robustly (strip NUL bytes)
    dt <- safe_fread_file(full_path)
    
    # append to merged file
    if (first_write) {
      fwrite(dt, merged_path)
      first_write <- FALSE
    } else {
      fwrite(dt, merged_path, append = TRUE)
    }
    
    file.remove(full_path)
  }
  
  # now de-duplicate
  dt_all <- safe_fread_file(merged_path)
  timestamp_cols <- grep("TIMESTAMP|DATE|time", names(dt_all),
                         ignore.case = TRUE, value = TRUE)

  if (length(timestamp_cols) > 0) {
    dt_all <- unique(dt_all, by = timestamp_cols)
  } else {
    dt_all <- unique(dt_all)
  }
  
  fwrite(dt_all, merged_path)
  rm(dt_all); gc()
}

Processing: US-Wi0

Warning message in fread(cmd = cmd, showProgress = FALSE, fill = TRUE):
“Previous fread() session was not cleaned up properly. Cleaned up ok at the beginning of this fread() call.”
Processing: ZA-Kru

Processing: US-xWR

Processing: AU-ASM

Processing: CA-Gro

Processing: JP-BBY

Processing: US-Srr

Processing: US-Fwf

Processing: CN-Cng

Processing: FI-Ken

Processing: US-Ho2

Processing: US-RGo

Processing: US-CS4

Processing: US-ASM

Processing: CR-Fsc

Processing: GL-ZaF

Processing: AU-Rob

Processing: FR-Lam

Processing: AU-Wom

Processing: US-SRM

Processing: NL-Loo

Warning message in fread(cmd = cmd, showProgress = FALSE, fill = TRUE):
“Stopped early on line 35090. Expected 166 fields but found 234. Consider fill=234 or even more based on your knowledge of the input file. Use fill=Inf for reading the whole file for detecting the number of fields. First discarded non-empty line: <<199601010000,199601010030,-9999,-9999,-2.104,-2.104,2,0,-9999,-9999,0,0,2,-999

ERROR: Error in fwrite(dt, merged_path): Input/output error: 'merged_sites/US-Ro2_merged.csv'


In [11]:
outdir <- "merged_sites"

In [12]:
existing   <- list.files(outdir, pattern = "_merged\\.csv$", full.names = FALSE)
sites_done <- str_extract(existing, "[A-Z]{2}-[A-Za-z0-9]+")
sites_to_do <- setdiff(unique_sites, sites_done)

sites_done
sites_to_do  # this should include US-Ro2 and the ones after it in the original loop

[1] "AR-Bal" "AR-CCa" "AR-SLu" "AR-TF1" "AR-Vir" "AT-Neu" "AU-ASM" "AU-Cpr"
  [9] "AU-Dry" "AU-Emr" "AU-Fog" "AU-GWW" "AU-Lit" "AU-Lox" "AU-Rgf" "AU-Rig"
 [17] "AU-Rob" "AU-Wac" "AU-Wom" "BE-Lcr" "BE-Lon" "BE-Maa" "BE-Vie" "BR-Sa3"
 [25] "CA-Ca1" "CA-Ca2" "CA-DB2" "CA-DBB" "CA-DSM" "CA-EM1" "CA-Gro" "CA-HPC"
 [33] "CA-MA2" "CA-MA3" "CA-Mer" "CA-NS1" "CA-NS2" "CA-NS4" "CA-NS5" "CA-NS6"
 [41] "CA-NS7" "CA-Obs" "CA-SMC" "CA-TP1" "CA-TP4" "CA-TPD" "CD-Ygb" "CH-Aws"
 [49] "CH-Dav" "CH-Fru" "CH-Oe1" "CL-SDF" "CL-SDP" "CN-Cng" "CN-Dan" "CN-Din"
 [57] "CN-Ha2" "CN-Qia" "CR-Fsc" "CZ-BK1" "CZ-KrP" "CZ-Lnz" "CZ-RAJ" "CZ-wet"
 [65] "DE-Akm" "DE-Geb" "DE-Gri" "DE-Hai" "DE-Har" "DE-Hdn" "DE-Hzd" "DE-Lkb"
 [73] "DE-Rns" "DE-RuS" "DE-Seh" "DE-SfS" "DE-Zrk" "DK-Fou" "DK-Gds" "DK-RCW"
 [81] "DK-Skj" "DK-Sor" "ES-LJu" "ES-LM2" "ES-LMa" "ES-Pdu" "FI-Hyy" "FI-Ken"
 [89] "FI-Let" "FI-Qvd" "FI-Sod" "FR-Aur" "FR-Bil" "FR-EM2" "FR-FBn" "FR-Fon"
 [97] "FR-Gri" "FR-Hes" "FR-Lam" "FR-Lqu" "FR-Mej" "FR-Pue" "GF-Guy" "GH-Ank"
[105] "GL-Dsk" "GL-ZaF" "GL-ZaH" "IE-CaN" "IE-Cra" "IE-Gwr" "IL-Yat" "IT-BCi"
[113] "IT-BFt" "IT-BsB" "IT-Cp2" "IT-La2" "IT-Lav" "IT-MBo" "IT-Noe" "IT-Ren"
[121] "IT-SR2" "IT-SRo" "JP-BBY" "MY-PSO" "NL-Loo" "NO-Hur" "PA-SPn" "RU-Cok"
[129] "RU-Fy3" "RU-Ha1" "RU-SkP" "RU-Tks" "SE-Deg" "SE-Htm" "SE-Svb" "US-A74"
[137] "US-AR2" "US-ASM" "US-Bar" "US-BRG" "US-BZF" "US-BZo" "US-CAK" "US-CdM"
[145] "US-CF1" "US-CGG" "US-Cop" "US-CRT" "US-CS4" "US-CS5" "US-CS6" "US-DFC"
[153] "US-DFK" "US-DS1" "US-DS3" "US-Elm" "US-Esm" "US-Fmf" "US-Fuf" "US-Fwf"
[161] "US-GBT" "US-HB1" "US-HB2" "US-Hn3" "US-Ho1" "US-Ho2" "US-IB2" "US-ICt"
[169] "US-Jo2" "US-KFS" "US-KPL" "US-KS2" "US-LA3" "US-LS2" "US-Me1" "US-Me2"
[177] "US-Me4" "US-Me5" "US-Me6" "US-MMS" "US-MN2" "US-MN3" "US-Mo1" "US-MOz"
[185] "US-MtB" "US-Myb" "US-NC1" "US-NC4" "US-Ne1" "US-Ne2" "US-NGC" "US-NR1"
[193] "US-Oho" "US-ORv" "US-PAS" "US-PFb" "US-PFc" "US-PFd" "US-PFh" "US-PFn"
[201] "US-PFr" "US-PFt" "US-Pnp" "US-PSL" "US-RGF" "US-RGo" "US-RGW" "US-Ro2"
[209] "US-Ro4" "US-Ro6" "US-RRC" "US-Rwe" "US-Seg" "US-SRM" "US-Srr" "US-Ton"
[217] "US-Tw1" "US-Tw2" "US-Tw3" "US-Tw5" "US-Twt" "US-UM3" "US-UMB" "US-UMd"
[225] "US-UTJ" "US-UTP" "US-UTV" "US-Vcm" "US-Whs" "US-Wi0" "US-Wi1" "US-Wi3"
[233] "US-Wi4" "US-Wi5" "US-Wi6" "US-Wi8" "US-Wi9" "US-xBA" "US-xCP" "US-xDJ"
[241] "US-xDL" "US-xHE" "US-xJE" "US-xJR" "US-xMB" "US-xRN" "US-xSB" "US-xSC"
[249] "US-xSE" "US-xSR" "US-xST" "US-xUN" "US-xWD" "US-xWR" "ZA-Kru"

[1] "CA-SF2" "US-UC2" "CA-Qfo" "FR-LGt" "US-ARM" "CA-TP3" "IT-PT1" "US-Syv"
  [9] "DE-Hte" "ES-LM1" "ES-Amo" "AU-Cum" "SE-Nor" "GL-NuF" "IT-Bsn" "CL-ACF"
 [17] "CA-Man" "PA-SPs" "US-SRS" "US-Wjs" "US-RGB" "US-Kon" "US-WCr" "SJ-Blv"
 [25] "CA-SF1" "US-Sag" "US-Ro3" "CA-SCC" "CA-ARB" "DE-RuC" "RU-Sam" "US-A39"
 [33] "IT-Ro1" "US-xBL" "US-ARc" "IT-CA3" "US-PFe" "US-xTR" "US-xCL" "US-Rwf"
 [41] "CA-Qc2" "UK-AMo" "CH-Cha" "CG-Tch" "US-Me3" "DE-Spw" "US-PFa" "US-Rpf"
 [49] "US-Vcp" "US-Prr" "US-SP1" "US-AR1" "US-UTB" "DE-RuW" "CZ-Stn" "IT-MtM"
 [57] "US-OWC" "SE-Sto" "AU-TTE" "ES-Ln2" "CA-TP2" "DK-Vng" "CH-Oe2" "DE-RuR"
 [65] "US-GLE" "US-MN1" "CN-Du3" "US-BZB" "CN-Sw2" "US-SRG" "US-Tw4" "RU-Fyo"
 [73] "FI-Var" "IT-Niv" "DE-HoH" "US-Sta" "IE-CaC" "CA-NS3" "US-Slt" "DE-Msr"
 [81] "CA-SF3" "CH-Lae" "CA-EM2" "US-xKA" "ES-LgS" "CA-SCB" "US-PFm" "AU-Adr"
 [89] "US-PFg" "AU-Boy" "US-xAB" "US-xHA" "US-Wi7" "US-xTA" "CN-Du2" "DE-Kli"
 [97] "AU-RDF" "US-YK2" "BE-Bra" "CZ-BK2" "US-CRK" "US-Snf" "US-ICh" "IT-TrF"
[105] "ZM-Mon" "US-xBN" "BE-Dor" "CA-LP1" "US-xSL" "US-Mo3" "US-CS8" "AU-Stp"
[113] "ES-Agu" "IT-Isp" "US-UC1" "JP-SMF" "EE-Pts" "US-CF4" "US-Hn2" "RU-Che"
[121] "US-Rms" "IT-CA1" "US-xNG" "IT-MtP" "US-Ne3" "US-CF3" "US-DS2" "US-Cst"
[129] "US-xNQ" "AU-Ade" "US-Rls" "IT-Tor" "ES-Abr" "US-Var" "FI-Sii" "AU-DaS"
[137] "US-CS3" "CA-ARF" "US-Dmg" "US-StJ" "US-PFL" "FI-Lom" "US-EML" "US-Rws"
[145] "US-CF2" "US-Blo" "US-Goo" "US-Lin" "US-PFk" "IT-Col" "US-RGA" "IT-Ro2"
[153] "NL-Hor" "US-BZS" "AR-CCg" "FR-Tou" "US-KS1" "US-KS3" "CA-ER1" "US-Bi1"
[161] "US-Ses" "AU-Tum" "US-Ha1" "CA-Cbo" "US-Wi2" "BR-Sa1" "US-A37" "US-Jo1"
[169] "IT-Cpz" "US-Fcr" "CA-Mtk" "FR-LBr" "US-EA5" "US-Ro5" "US-PSH" "DE-Tha"
[177] "US-PFp" "PE-QFR" "CH-Frk" "US-xAE" "US-Wkg" "ES-Cnd" "SD-Dem" "CN-Cha"
[185] "US-Bi2" "DE-Lnf" "IT-Lsn" "IT-CA2" "BR-CST" "SN-Dhr" "US-ARb" "AU-Gin"
[193] "AU-Whr" "FR-CLt" "US-WPT" "CA-MA1" "US-Hsm" "FI-Jok" "US-CS2" "DE-SfN"
[201] "CA-Oas" "US-NC3" "US-LWW" "US-xSJ" "US-xRM" "US-SHC" "SE-Ros" "US-CS1"
[209] "AU-War" "US-UTD" "US-xDS" "JP-MBF" "DE-Obe" "AU-Lon" "US-EA6" "US-xYE"
[217] "MX-Tes" "US-Akn" "US-KLS" "SJ-Adv" "CA-BOU" "SE-St1" "US-ASH" "US-EDN"
[225] "US-Sne" "RU-Fy2" "US-Los" "US-xKZ" "US-HWB" "RU-Vrk" "US-EvM" "US-SRC"
[233] "US-HVs" "US-xBR" "US-DPW" "US-PFj" "CA-CF1" "US-ICs" "US-PFq" "US-xTL"
[241] "US-xUK" "US-Mpj" "US-Cms" "US-SSH" "FR-Lus" "US-ALQ" "US-xNW" "US-ONA"
[249] "CN-HaM" "US-HB3" "US-UTE" "DK-Eng" "AR-TF2" "BR-Npw" "US-xDC" "US-xGR"
[257] "US-NGB" "US-Ivo" "AU-Ync" "US-A32" "US-Atq" "US-HB4" "AU-How" "US-Ro1"
[265] "AU-DaP" "US-Mo2" "US-xML"

In [13]:
safe_fread_file <- function(path) {
  cmd <- sprintf("tr -d '\\000' < %s", shQuote(path))
  data.table::fread(cmd = cmd, showProgress = FALSE, fill = TRUE)
}

data.table::setDTthreads(8)

for (site in sites_to_do) {
  message("=== Processing site: ", site, " ===")
  
  site_files  <- csv_files[site_ids == site]
  merged_path <- file.path(outdir, paste0(site, "_merged.csv"))
  
  # in case US-Ro2_merged.csv exists but is corrupted/partial
  if (file.exists(merged_path)) file.remove(merged_path)
  
  first_write <- TRUE
  
  for (f in site_files) {
    unzip(zipfile, files = f, exdir = tempdir(), overwrite = TRUE)
    full_path <- file.path(tempdir(), f)
    
    dt <- safe_fread_file(full_path)
    
    data.table::fwrite(dt, merged_path,
                       append = !first_write)
    first_write <- FALSE
    
    file.remove(full_path)
  }
  
  dt_all <- safe_fread_file(merged_path)
  ts_cols <- grep("TIMESTAMP|DATE|time", names(dt_all),
                  ignore.case = TRUE, value = TRUE)
  if (length(ts_cols) > 0) {
    dt_all <- unique(dt_all, by = ts_cols)
  } else {
    dt_all <- unique(dt_all)
  }
  data.table::fwrite(dt_all, merged_path)
  rm(dt_all); gc()
}

=== Processing site: CA-SF2 ===



=== Processing site: US-UC2 ===

=== Processing site: CA-Qfo ===

Warning message in data.table::fread(cmd = cmd, showProgress = FALSE, fill = TRUE):
“Stopped early on line 140259. Expected 226 fields but found 228. Consider fill=228 or even more based on your knowledge of the input file. Use fill=Inf for reading the whole file for detecting the number of fields. First discarded non-empty line: <<200301010000,200301010030,-9999,-9999,-7.385,-7.385,2,0,-9999,-9999,0,0,2,-9999,-9999,241.792,241.792,2,-9999,-9999,220.415,220.415,2,-9999,-9999,0.528,0.528,2,-9999,96.124,96.124,2,-9999,0.047,0.047,2,-9999,2.779,2.779,2,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,0.663125,3,0.964118,-9999,-9999,-9999,-9999,-9999,-9999,-1.885,3,-2.73864,-9999,-9999,-9999,-9999,-9999,-9999,55,3,1,2.42933,0.9465,3,3,-9999,-9999,-999>>”
=== Processing site: FR-LGt ===

=== Processing site: US-ARM ===

Warning message in data.table:

In [39]:
system("df -h .")                 # free space on the filesystem of your home dir
system("du -sh merged_sites")     # how big the output folder is

In [ ]:
View(EFP_site)

SITE_ID,siteID,uWUE,ETmax,precipAvail,Gavail,GSmax,CO2avail,G1,EF,⋯,Rb,Rbmax,aCUE,TZ,nyears,SITE_NAME,LOCATION_LAT,LOCATION_LONG,LOCATION_ELEV,IGBP
<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<chr>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<int>,<int>,<chr>,<dbl>,<dbl>,<dbl>,<chr>
AT-Neu,AT-Neu,4.4531612,0.26577914,yes,yes,0.0149402500,yes,1.5027279,0.7901329,⋯,15.385701,21.762126,0.45201781,1,19,Neustift,47.11670,11.317500,970.00,GRA
BE-Lon,BE-Lon,2.9829694,0.23079557,yes,yes,0.0152947875,yes,0.9652484,0.6458370,⋯,3.343010,6.216584,0.47073279,1,21,Lonzee,50.55162,4.746234,157.00,CRO
BR-CST,BR-CST,3.7205283,0.18012509,yes,yes,0.0005544469,yes,-1.0542127,0.2228430,⋯,NA,NA,NA,-3,2,NA,-7.96820,-38.384200,468.00,PFT_DNF
BR-Npw,BR-Npw,1.9239528,0.35318931,yes,yes,0.0004802108,yes,-0.8785709,0.7826637,⋯,NA,NA,NA,-3,5,Northern Pantanal Wetland,-16.49800,-56.412000,120.00,WSA
CA-Ca1,CA-Ca1,6.0950839,0.12927937,yes,yes,0.0064543504,yes,0.3026107,0.3228162,⋯,3.486950,8.717625,0.77525906,-8,15,NA,49.86730,-125.333600,303.06,PFT_ENF
CA-Ca2,CA-Ca2,2.9547596,0.15292241,yes,yes,0.0055918070,yes,1.7197547,0.3612395,⋯,NA,NA,NA,-8,12,NA,49.87050,-125.290900,174.01,PFT_ENF
CA-Cbo,CA-Cbo,2.7412119,0.25932713,yes,yes,0.0134334459,yes,1.7711470,0.4188698,⋯,6.278615,12.304131,-0.35495133,-5,27,NA,44.31670,-79.933300,215.72,PFT_DBF
CA-DB2,CA-DB2,0.8900540,0.13428289,yes,yes,0.0050754978,yes,5.8545911,0.3734183,⋯,NA,NA,NA,-8,2,NA,49.11900,-122.995100,9.21,PFT_WET
CA-DBB,CA-DBB,0.7071730,0.17592585,yes,yes,0.0057465202,yes,5.9821966,0.4484511,⋯,NA,NA,NA,-8,7,NA,49.12930,-122.984900,10.21,PFT_WET


In [ ]:
#select each unique IGBP and SITE_ID which has more than 5 yearn in EFP_all data
EFP_summary <- EFP_site %>%
  group_by(IGBP, SITE_ID) %>%
  #summarise(years_count = n_distinct(YEAR)) %>%
  filter(nyears >= 5)

In [ ]:
View(EFP_summary)

SITE_ID,siteID,uWUE,ETmax,precipAvail,Gavail,GSmax,CO2avail,G1,EF,⋯,Rb,Rbmax,aCUE,TZ,nyears,SITE_NAME,LOCATION_LAT,LOCATION_LONG,LOCATION_ELEV,IGBP
<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<chr>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<int>,<int>,<chr>,<dbl>,<dbl>,<dbl>,<chr>
AT-Neu,AT-Neu,4.4531612,0.26577914,yes,yes,0.0149402500,yes,1.5027279,0.7901329,⋯,15.385701,21.762126,0.45201781,1,19,Neustift,47.11670,11.317500,970.00,GRA
BE-Lon,BE-Lon,2.9829694,0.23079557,yes,yes,0.0152947875,yes,0.9652484,0.6458370,⋯,3.343010,6.216584,0.47073279,1,21,Lonzee,50.55162,4.746234,157.00,CRO
BR-Npw,BR-Npw,1.9239528,0.35318931,yes,yes,0.0004802108,yes,-0.8785709,0.7826637,⋯,NA,NA,NA,-3,5,Northern Pantanal Wetland,-16.49800,-56.412000,120.00,WSA
CA-Ca1,CA-Ca1,6.0950839,0.12927937,yes,yes,0.0064543504,yes,0.3026107,0.3228162,⋯,3.486950,8.717625,0.77525906,-8,15,NA,49.86730,-125.333600,303.06,PFT_ENF
CA-Ca2,CA-Ca2,2.9547596,0.15292241,yes,yes,0.0055918070,yes,1.7197547,0.3612395,⋯,NA,NA,NA,-8,12,NA,49.87050,-125.290900,174.01,PFT_ENF
CA-Cbo,CA-Cbo,2.7412119,0.25932713,yes,yes,0.0134334459,yes,1.7711470,0.4188698,⋯,6.278615,12.304131,-0.35495133,-5,27,NA,44.31670,-79.933300,215.72,PFT_DBF
CA-DBB,CA-DBB,0.7071730,0.17592585,yes,yes,0.0057465202,yes,5.9821966,0.4484511,⋯,NA,NA,NA,-8,7,NA,49.12930,-122.984900,10.21,PFT_WET
CA-Gro,CA-Gro,2.5516064,0.19145826,yes,yes,0.0111447820,yes,2.5615283,0.3362806,⋯,NA,NA,NA,-5,12,"Ontario - Groundhog River, Boreal Mixedwood Forest",48.21670,-82.155600,340.00,MF
CA-LP1,CA-LP1,2.4646896,0.11205929,yes,yes,0.0060691962,yes,2.9510491,0.2544924,⋯,NA,NA,NA,-8,16,NA,55.11190,-122.841400,751.00,PFT_ENF
